In [ ]:

# Install required NLP libraries
!pip -q install vaderSentiment wordcloud

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud
from collections import Counter


In [ ]:

# Upload dataset
uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)
df.head()


In [ ]:

# Inspect dataset
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:

# Detect important columns automatically
text_candidates = [
    'text', 'tweet', 'tweet_text', 'post', 'content', 'comment',
    'review', 'message', 'sentiment_text'
]

date_candidates = [
    'date', 'created_at', 'timestamp', 'datetime', 'time',
    'posted_at', 'created'
]

text_col = next(
    (c for c in df.columns if c.lower().strip() in text_candidates),
    None
)

date_col = next(
    (c for c in df.columns if c.lower().strip() in date_candidates),
    None
)

if text_col is None:
    object_cols = df.select_dtypes(include='object').columns.tolist()
    if object_cols:
        text_col = max(object_cols, key=lambda c: df[c].astype(str).str.len().mean())

print("Text column:", text_col)
print("Date column:", date_col)


In [ ]:

# Clean data
df = df.drop_duplicates().copy()
df[text_col] = df[text_col].fillna('').astype(str)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df[text_col].apply(clean_text)
df = df[df['clean_text'].str.len() > 0].copy()

print("Cleaned dataset shape:", df.shape)
df[[text_col, 'clean_text']].head()


In [ ]:

# Remove common stop words
stop_words = {
    'a','an','the','and','or','but','if','then','than','so','of','to','in',
    'on','for','with','at','by','from','as','is','it','this','that','be',
    'are','was','were','am','i','you','he','she','we','they','my','your',
    'our','their','me','him','her','us','them','do','does','did','have',
    'has','had','will','would','can','could','should','not','very','just',
    'about','into','over','after','before','more','most','some','any'
}

def remove_stopwords(text):
    return ' '.join(word for word in text.split() if word not in stop_words)

df['analysis_text'] = df['clean_text'].apply(remove_stopwords)
df[['clean_text', 'analysis_text']].head()


In [ ]:

# Sentiment classification using VADER
analyzer = SentimentIntensityAnalyzer()

def classify_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_score'] = df['clean_text'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)
df['sentiment'] = df['clean_text'].apply(classify_sentiment)

print(df['sentiment'].value_counts())
df[[text_col, 'sentiment_score', 'sentiment']].head(10)


In [ ]:

# Sentiment distribution
sentiment_counts = df['sentiment'].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values)
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Posts')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 7))
plt.pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Sentiment Distribution (%)')
plt.tight_layout()
plt.show()


In [ ]:

# Generate word clouds for all sentiments
for sentiment in ['Positive', 'Negative', 'Neutral']:
    text = ' '.join(
        df.loc[df['sentiment'] == sentiment, 'analysis_text']
    )

    if text.strip():
        wordcloud = WordCloud(
            width=1000,
            height=500,
            background_color='white',
            max_words=100
        ).generate(text)

        plt.figure(figsize=(12, 6))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'{sentiment} Sentiment Word Cloud')
        plt.show()


In [ ]:

# Find most frequent terms
all_words = ' '.join(df['analysis_text']).split()
word_counts = Counter(all_words)

top_words = pd.DataFrame(
    word_counts.most_common(15),
    columns=['Word', 'Frequency']
)

print(top_words)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_words, x='Frequency', y='Word')
plt.title('Top 15 Frequently Used Terms')
plt.tight_layout()
plt.show()


In [ ]:

# Sentiment trends over time
if date_col is not None:
    df['date_parsed'] = pd.to_datetime(df[date_col], errors='coerce')
    trend_df = df.dropna(subset=['date_parsed']).copy()

    if len(trend_df) > 0:
        trend_df['date'] = trend_df['date_parsed'].dt.date

        daily_sentiment = (
            trend_df.groupby(['date', 'sentiment'])
            .size()
            .unstack(fill_value=0)
        )

        daily_sentiment.plot(figsize=(14, 6), marker='o')
        plt.title('Sentiment Trend Over Time')
        plt.xlabel('Date')
        plt.ylabel('Number of Posts')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print("Date range:", trend_df['date_parsed'].min(), "to", trend_df['date_parsed'].max())
    else:
        print("Date column was detected, but valid dates could not be parsed.")
else:
    print("No date column was detected. Time-based sentiment analysis is skipped.")


In [ ]:

# Average sentiment score
print("Average sentiment score:", round(df['sentiment_score'].mean(), 4))

print("\nAverage sentiment score by category:")
print(
    df.groupby('sentiment')['sentiment_score']
      .agg(['count', 'mean'])
      .sort_values('mean', ascending=False)
)

print("\nPositive percentage:",
      round((df['sentiment'].eq('Positive').mean()) * 100, 2), "%")

print("Negative percentage:",
      round((df['sentiment'].eq('Negative').mean()) * 100, 2), "%")

print("Neutral percentage:",
      round((df['sentiment'].eq('Neutral').mean()) * 100, 2), "%")


In [ ]:

# Automatically generate business insights and recommendations
positive_pct = df['sentiment'].eq('Positive').mean() * 100
negative_pct = df['sentiment'].eq('Negative').mean() * 100
neutral_pct = df['sentiment'].eq('Neutral').mean() * 100

print("KEY INSIGHTS")
print("-" * 50)

if positive_pct >= negative_pct:
    print(f"1. Positive sentiment ({positive_pct:.1f}%) is higher than negative sentiment ({negative_pct:.1f}%).")
    print("   Maintain successful campaigns and customer experiences that are generating positive reactions.")
else:
    print(f"1. Negative sentiment ({negative_pct:.1f}%) is higher than positive sentiment ({positive_pct:.1f}%).")
    print("   Prioritize complaint analysis, customer support, and reputation-management activities.")

print(f"2. Neutral sentiment represents {neutral_pct:.1f}% of posts, indicating an opportunity to increase meaningful engagement.")

if negative_pct > 30:
    print("3. A relatively high negative-sentiment share suggests that recurring complaints should be investigated.")
else:
    print("3. Negative sentiment is not dominant, but negative posts should still be monitored for emerging issues.")

print("\nRECOMMENDATIONS")
print("-" * 50)
print("• Monitor negative posts and identify recurring complaint themes.")
print("• Respond quickly to customer concerns and service-related complaints.")
print("• Use positive themes in future marketing campaigns.")
print("• Track sentiment regularly to detect changes in customer perception.")
print("• Combine sentiment results with engagement metrics for stronger marketing decisions.")


In [ ]:

# Final project summary
summary = {
    'Total Posts Analyzed': len(df),
    'Positive Posts': int((df['sentiment'] == 'Positive').sum()),
    'Negative Posts': int((df['sentiment'] == 'Negative').sum()),
    'Neutral Posts': int((df['sentiment'] == 'Neutral').sum()),
    'Average Sentiment Score': round(df['sentiment_score'].mean(), 4)
}

summary_df = pd.DataFrame(
    summary.items(),
    columns=['Metric', 'Value']
)

summary_df
